# 13 · SkewerJAX and fake_spectra: where the time goes

How long the two codes take to compute the same spectra, stage by stage, on one target: **TNG50-2** (snapshot 25,
$z = 3.0$, 1.19 billion gas cells, 349 GB), a regular **grid of $16 \times 16$ skewers along $y$** (256 skewers), fake_spectra's
pixels of 1 km/s, and the lines **$\mathrm{Ly}\alpha$** and **Si III 1206**. Five entries, each timed by stage on the same machine:

- **fake_spectra** (native, kernel `skewerjax-fs`, its OpenMP on every core);
- **SkewerJAX default** and **SkewerJAX matched** (every option at fake_spectra's value, notebook 12), on the **CPU** and on
  the **GPU**.

The stages: **setup** (the skewers), **I/O** (the snapshot reads), **cell selection** (the cells within reach of the
skewers), **gas physics** (temperatures, ion fractions), **pieces** (the deposition of the cells along the skewers) and the
**$\tau$ sum**. fake_spectra searches its particles, deposits them and sums $\tau$ inside its C engine: its selection and pieces
are in its $\tau$. What each code computes, and how close they are, is in notebook 12; how the codes scale with the number of
skewers and the snapshot size, with the records of an earlier version, in notebook 14.

In [ ]:
import os
N_CPU = os.cpu_count()                                   # logical CPUs of this machine
os.environ["OMP_NUM_THREADS"] = str(N_CPU)               # fake_spectra's OpenMP threads (its C engine)
import skewerjax.config as config                        # imports neither JAX nor the package: the devices come first
config.configure_devices(
    n_cpu_devices=1,          # the CPU as one JAX device: XLA's own threads use every core (sharding the skewers over
                              # N_CPU host devices measured no faster for these kernels; config.devices("cpu") either way)
    platforms="cuda,cpu",     # the GPU next to the CPU (config.devices("gpu") is empty without one)
)
import contextlib, io, json, logging, tempfile
logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import add_panel_letters, plot_stage_speedups, plot_stage_times, use_backend, use_style
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()        # paper style: Tol's colour-blind-safe "bright" colours and colour maps, minimum font sizes

CPU, GPU = config.devices("cpu"), config.devices("gpu")
cpu_model = next(l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name"))
ram_GB = next(int(l.split()[1]) for l in open("/proc/meminfo") if l.startswith("MemTotal")) / 1e6
print(f"CPU: {cpu_model}, {N_CPU} logical CPUs, {ram_GB:.0f} GB RAM; JAX CPU devices {CPU}")
print(f"GPU: {[d.device_kind for d in GPU] or 'none'}; precision {config.get_precision()} (the package's default)")

## The set-up

The skewers are in **box mode** (`SkewerBundle.axis_aligned`): parallel skewers across the box share one path coordinate,
so one $a(s)$, solved once in float64 on the host with $a = a_{\rm snap}$ at the middle of the path, one frame and one
pixel grid. The pixels are fake_spectra's: $n_{\rm bins} = \lfloor v_{\max}/1\ \mathrm{km/s}\rfloor$,
$dv = v_{\max}/n_{\rm bins}$, $v_{\max} = (aH)_{\rm snap}L$. SkewerJAX samples each path every 0.25 km/s.

**Protocol.** Each stage runs `1 + N_RUNS` times: run 0 is the warm-up (JIT compilation, caches) and is reported apart;
the bars are the medians of the timed runs. Every JAX result is pulled to the host (`np.asarray`), which waits until it is
ready (`block_until_ready`). fake_spectra compiles nothing and reads more than the RAM holds, so all its runs are timed.
Everything is in float32 (the package's precision; fake_spectra's C engine works in double).

In [ ]:
from skewerjax.core import LineBundle, SkewerBundle, SnapshotFrame, cell_pieces, compute_optical_depth
from skewerjax.simulations import ArepoHDF5Reader, gas_to_cells, select_cells_near_paths
from skewerjax.comparison.benchmark import RunClock, StageTimer
from skewerjax.utils.paths import BENCHMARK_RECORDS_DIR, DATA_DIR as DATA   # $SKEWERJAX_DATA (docs/installation.md)

In [ ]:
lines = LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id="HI-LyA")
SPECIES = list(lines.species_keys)

In [ ]:
SIM, SNAP, AXIS, SPACING_KMS, N_GRID = "TNG50-2", 25, 2, 0.25, 16     # along y (axis 2); points every 0.25 km/s; 16 x 16 skewers
N_RUNS, N_RUNS_FS = 3, 3                                               # timed runs (SkewerJAX after a warm-up; fake_spectra)
reader = ArepoHDF5Reader(f"{DATA}/{SIM}", SNAP)
H = reader.header
L, AH, a_snap = H.box_cMpc, H.velfac_kms_per_cMpc, H.a                 # [cMpc], (aH)_snap [km/s per cMpc]
nbins, dv = H.pixel_grid(1.0)                                          # fake_spectra's pixels
v = (np.arange(nbins) + 0.5) * dv                                      # [km/s], over [0, v_max)
V_PERIOD = nbins * dv
print(f"{SIM}, z = {H.redshift:.3f}, L = {L:.3f} cMpc, {H.n_gas:,} gas cells in {H.n_files} files; {nbins} pixels of {dv:.6f} km/s")

In [ ]:
clock = RunClock()                                    # entry -> stage -> seconds per run (run 0: the warm-up)
CPU_DEF, CPU_MAT, GPU_DEF, GPU_MAT, FS = (
    "SkewerJAX default, CPU", "SkewerJAX matched, CPU", "SkewerJAX default, GPU", "SkewerJAX matched, GPU", "fake_spectra"
)
g = np.arange(N_GRID) * L / N_GRID                    # x, z = i L / 16 [cMpc]
X, Z = (c.ravel() for c in np.meshgrid(g, g, indexing="ij"))
for run in range(1 + N_RUNS):
    with clock.time(CPU_DEF, "setup", run):
        bundle = SkewerBundle.axis_aligned(
            np.column_stack([X, np.zeros_like(X), Z]),   # a point of each skewer [cMpc] (its y is ignored)
            AXIS,                                          # every skewer along y, across the box once
            L,
            SPACING_KMS / AH,                              # the point spacing [cMpc]: 0.25 km/s of Hubble flow
            H.cosmology(),
            a_ref=a_snap,                                  # a(s) = a_snap at the middle of the path
        )
points = bundle.host["points"]                        # (256, N, 3) [cMpc], float64
print(f"{bundle.N_sk} skewers of {bundle.N_ppsk:,} points; a from {bundle.host['scale_factors'][0, 0, 0]:.5f} to "
      f"{bundle.host['scale_factors'][0, 0, -1]:.5f} (a_snap = {a_snap:.5f}); setup {clock.medians()[CPU_DEF]['setup'] * 1e3:.0f} ms")

## fake_spectra

`Spectra` on the same 256 sightlines (an axis and a point in its units, ckpc/$h$) and the same pixel width, then `get_tau`
for each line. Its stages are timed from outside, without editing it: the construction of `Spectra` (**setup**), every call
of its snapshot reader `snapshot_set.get_data` (**I/O**), every call of its C engine `_do_interpolation_work` (**$\tau$**: the
particles near the sightlines, their deposition and the sum), and the rest of `get_tau` (**gas physics**: temperatures,
neutral fractions, the CLOUDY table, on every particle of every file). It reads the snapshot once per line.

In [ ]:
from fake_spectra.spectra import Spectra
WORK = tempfile.mkdtemp(prefix="fs_timing_")
cofm = np.column_stack([X, np.zeros_like(X), Z]) / H.code_length_cMpc    # [ckpc/h]
axis = np.full(X.size, AXIS, dtype=np.int32)                              # 2 = y
fs_runs = []
for run in range(N_RUNS_FS):
    T = StageTimer()
    with contextlib.redirect_stdout(io.StringIO()):                      # its progress messages, per file
        with T.stage("setup"):
            fs = Spectra(SNAP, f"{DATA}/{SIM}", cofm.copy(), axis.copy(), res=1.0, savefile="t.hdf5", savedir=WORK,
                         reload_file=True, quiet=True)
        with T.patched((fs.snapshot_set, "get_data", "io"), (fs, "_do_interpolation_work", "tau")):
            with T.stage("physics"):
                tau_fs = np.stack([np.asarray(fs.get_tau("H", 1, 1215)), np.asarray(fs.get_tau("Si", 3, 1206))])
    fs_runs.append(dict(T.t))
    print(f"run {run}: " + ", ".join(f"{k} {t:.1f} s" for k, t in T.t.items() if t > 0))
for stage in ("setup", "io", "selection", "physics", "pieces", "tau"):
    clock.add(FS, stage, [r[stage] for r in fs_runs], warmup=False)
print(f"{os.environ['OMP_NUM_THREADS']} OpenMP threads; kernel {fs.kernel_int} (0 = top hat); {fs.nbins} pixels of {fs.dvbin:.6f} km/s")

## SkewerJAX default, CPU

The calls of notebook 12's default block, every option written, on the 256 skewers. The I/O and the selection are one call
(`select_cells_near_paths`): a thread per block of every file, read through memory maps, a prefilter of distance bounds
on a $256^3$ grid around the skewers, and the exact rule (radius $2.5\,V^{1/3} + ds/2$ from a path point) on the
candidates; its own timings split the two (each stage's share of the wall time, as the threads overlap them).

In [ ]:
for run in range(1 + N_RUNS):
    gas, info = select_cells_near_paths(
        reader, points,
        mode="voronoi",          # the radius of the Voronoi deposition (a superset of the top hat's: one selection for both modes)
        max_rows=4_000_000,      # cells per block, one block per thread
        workers=-1,              # threads: every core
        backend="cpu",
        return_info=True,
    )
    clock.record(CPU_DEF, "io", run, info["timings"]["read"] + info["timings"]["read_selected"])
    clock.record(CPU_DEF, "selection", run, info["timings"]["prefilter"] + info["timings"]["query"])
    print(f"run {run}: {info['time_s']:.1f} s ({', '.join(f'{k} {t:.1f}' for k, t in info['timings'].items())}), "
          f"{info['n_selected']:,} of {info['n_gas']:,} cells ({info['n_candidates']:,} candidates), {info['threads']} threads")

In [ ]:
for run in range(1 + N_RUNS):
    with clock.time(CPU_DEF, "physics", run):
        cells = gas_to_cells(
            gas, SPECIES,
            mass_unit="amu",
            T_constants="codata",
            x_HI_model="stored_sf",
            cloudy_density="hydrogen",
            cloudy_interpolation="linear",
        )

In [ ]:
for run in range(1 + N_RUNS):
    with clock.time(CPU_DEF, "pieces", run):
        pieces = cell_pieces(
            cells, lines,
            skewer_bundle=bundle,
            deposition="voronoi",
            a_data=a_snap,
        )
print(f"pieces: {pieces.shape} (lines, skewers, pieces per skewer)")

In [ ]:
for run in range(1 + N_RUNS):
    with clock.time(CPU_DEF, "tau", run):
        tau_def_cpu = np.asarray(compute_optical_depth(      # np.asarray waits for the result
            pieces, v, lines,
            velocity_period=False,
            voigt="harris",                 # the default: Harris' expansion (fourth order for Si III), ExoJAX above a = 0.05
            detuning="exact",
            a_eff=None,
            hubble_flow="exact",
            aH_ref=None,
            b_constants="codata",
            pixel_average=False,
            tautail=None,
            resolution_tolerance=0.25,
            piece_chunk=2048,
            dtype=None,                     # the package's precision: float32
            devices=CPU,
        ))

## SkewerJAX matched, CPU

The same calls with every option at fake_spectra's value (notebook 12, the matched block). The skewers, the I/O and the
selection are shared with the default (the same cells). The sum is fake_spectra's per-cell scheme: the pixel average of
narrow cells and its early stop `tautail` (each cell walks out from its line centre until it adds less than $10^{-7}$),
evaluated on a window around every cell (`core.per_cell`) and batched over the 256 skewers.

In [ ]:
for stage in ("setup", "io", "selection"):
    clock.share(CPU_DEF, stage, CPU_MAT)
frame = SnapshotFrame(a_snap, AH, path_length=np.full(bundle.N_sk, L))    # one a and u = aH s + v_pec
for run in range(1 + N_RUNS):
    with clock.time(CPU_MAT, "physics", run):
        cells_m = gas_to_cells(
            gas, SPECIES,
            mass_unit="proton",
            T_constants="fake_spectra",
            x_HI_model="fake_sf",
            cloudy_density="fake_spectra",
            cloudy_interpolation="cubic",
        )
for run in range(1 + N_RUNS):
    with clock.time(CPU_MAT, "pieces", run):
        pieces_m = cell_pieces(
            cells_m, lines,
            points=points,
            frame=frame,
            deposition="tophat_chords",
            chord_weight=1 / 8,
            a_data=a_snap,
        )
print(f"pieces: {pieces_m.shape}")

In [ ]:
MATCHED_TAU = dict(
    velocity_period=V_PERIOD,
    voigt="harris",
    detuning="linear",
    a_eff=a_snap,
    hubble_flow="linear",
    aH_ref=AH,
    b_constants="fake_spectra",
    pixel_average=True,
    tautail=1e-7,
    resolution_tolerance=None,
)
for run in range(1 + N_RUNS):
    with clock.time(CPU_MAT, "tau", run):
        tau_mat_cpu = np.asarray(compute_optical_depth(pieces_m, v, lines, devices=CPU, **MATCHED_TAU))

## SkewerJAX default and matched on the GPU

The same calls with `devices=GPU`. The host stages (setup, I/O, selection, gas physics, pieces) are the CPU's: they are
the same calls, on the host, so their times are taken from the CPU entries; only the $\tau$ sum moves to the GPU.

In [ ]:
if GPU:
    for entry, cpu_entry in ((GPU_DEF, CPU_DEF), (GPU_MAT, CPU_MAT)):
        for stage in ("setup", "io", "selection", "physics", "pieces"):
            clock.share(cpu_entry, stage, entry)
    for run in range(1 + N_RUNS):
        with clock.time(GPU_DEF, "tau", run):
            tau_def_gpu = np.asarray(compute_optical_depth(
                pieces, v, lines,
                velocity_period=False, voigt="harris", detuning="exact", a_eff=None, hubble_flow="exact", aH_ref=None,
                b_constants="codata", pixel_average=False, tautail=None, resolution_tolerance=0.25, piece_chunk=2048,
                dtype=None,
                devices=GPU,
            ))
    for run in range(1 + N_RUNS):
        with clock.time(GPU_MAT, "tau", run):
            tau_mat_gpu = np.asarray(compute_optical_depth(pieces_m, v, lines, devices=GPU, **MATCHED_TAU))
else:
    print("no GPU visible to JAX: the GPU entries are left out")

## Fairness and correctness

The five entries compute the same 256 spectra of the same two lines on the same pixels, in float32 (fake_spectra in
double). The checks: the matched mode against fake_spectra (rms $\Delta F$ over the 256 skewers and their pixels), the GPU
against the CPU (to float32 rounding), and the default against the matched mode (the known difference of notebook 12).
**The I/O cache:** fake_spectra reads the fields of every particle once per line, more than the RAM holds, so the page
cache cannot keep them from one run to the next; its three runs here read for the same 62 s each (the baseline run of
the code before the optimisation, earlier the same day, 111 s: the same code, a different state of the disk and the cache).
SkewerJAX reads the positions, masses and densities of every cell (memory maps) and then the selected rows only: its
warm-up run, right after fake_spectra's reads, finds them mostly on the disk, its timed runs in the page cache (the
warm-up's I/O is printed below).

In [ ]:
rms = lambda a, b: float(np.sqrt(np.mean((np.exp(-np.asarray(a, float)) - np.exp(-np.asarray(b, float))) ** 2)))
mx = lambda a, b: float(np.max(np.abs(np.exp(-np.asarray(a, float)) - np.exp(-np.asarray(b, float)))))
for j, line in enumerate(lines.lines):
    print(f"{line.id:11s} matched (CPU) vs fake_spectra: rms dF {rms(tau_mat_cpu[j], tau_fs[j]):.1e}, max {mx(tau_mat_cpu[j], tau_fs[j]):.1e}"
          f" | default vs matched: rms dF {rms(tau_def_cpu[j], tau_mat_cpu[j]):.1e}")
    if GPU:
        print(f"{'':11s} GPU vs CPU: default rms dF {rms(tau_def_gpu[j], tau_def_cpu[j]):.1e}, matched {rms(tau_mat_gpu[j], tau_mat_cpu[j]):.1e}")
w, m = clock.warmup()[CPU_DEF], clock.medians()[CPU_DEF]
print(f"SkewerJAX I/O: warm-up run {w['io']:.1f} s (after fake_spectra's reads), timed runs {m['io']:.1f} s (page cache)")
print(f"JIT compilation (warm-up minus median), tau: " + ", ".join(f"{e} {c:.2f} s" for e, c in clock.compile_s("tau").items() if e != FS))
print()
print(clock.table())

## Where the time goes

The median wall time of one run, split into its stages (left: seconds; right: each stage's share). The JIT compilation of
the warm-up run is written after each bar; it is not in the bars.

In [ ]:
order = [FS, CPU_DEF, CPU_MAT] + ([GPU_DEF, GPU_MAT] if GPU else [])
med = clock.medians()
entries = {e: med[e] for e in order}
fig, axs = plt.subplots(1, 2, figsize=(22, 7.5), layout="constrained", gridspec_kw={"width_ratios": [1.25, 1]})
plot_stage_times(axs[0], entries, compile_s=clock.compile_s("tau"))
plot_stage_times(axs[1], entries, fraction=True)
axs[0].set_xlim(right=axs[0].get_xlim()[1] * 1.45)                  # room for the totals
axs[1].set_yticklabels([])
handles, labels = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside lower center", ncol=6, frameon=False, fontsize=15)
add_panel_letters(axs)
fig.suptitle("fake_spectra spends its run reading and processing every particle; SkewerJAX its pieces and its τ sum", fontsize=21)
plt.show()

## The speed-up

The total of every entry (filled), with its speed-up over fake_spectra, and the same entries before this optimisation
(hatched: one run of the code at commit `8b70ddd`, float64, the same target, SkewerJAX only: fake_spectra's code has not changed; `benchmarks/results/timing_baseline_20261107.json`).

In [ ]:
before = json.loads((BENCHMARK_RECORDS_DIR / "timing_baseline_20261107.json").read_text())
totals = clock.totals()
fig, ax = plt.subplots(figsize=(22, 6.5), layout="constrained")
plot_stage_speedups(
    ax, {e: totals[e] for e in order},
    reference=FS,
    before={e: sum(st.values()) for e, st in before["entries"].items() if e != FS},   # fake_spectra: the same code
)
ax.set_xlim(right=ax.get_xlim()[1] * 8)                     # room for the labels
ax.legend(loc="lower right", fontsize=15, frameon=False)
fig.suptitle("SkewerJAX now computes the 256 spectra faster than fake_spectra, in both modes and on both devices", fontsize=21)
plt.show()
for e in order:
    b = sum(before["entries"][e].values()) if e in before["entries"] and e != FS else None
    print(f"{e:24s} {totals[e]:7.1f} s: {totals[FS] / totals[e]:5.2f}x fake_spectra's speed" + (f", {b / totals[e]:5.1f}x faster than before ({b:.0f} s)" if b else ""))

## Discussion

**Where each code spends its time.** fake_spectra (203 s) spends 30 % of its run reading and 60 % on the gas physics
(temperatures, neutral fractions, the CLOUDY table) of *every* particle of every file, once per line; its C engine (the
particles near the sightlines, their deposition and the $\tau$ sum) takes 19 s, 9 %. Its cost follows the size of the snapshot,
not the number of skewers. SkewerJAX reads three datasets of every cell (2.1 s from the page cache), keeps the
4.48 million cells within reach of the skewers (0.37 % of 1.2 billion; 6.6 s) and computes the gas physics of those only
(0.8 s): its host stages take 10 s in all. The rest is the per-skewer work: the pieces (20 s default, 9.4 s matched) and the
$\tau$ sum (32 s and 23 s on the CPU, 4.5 s and 5.5 s on the GPU).

**What was optimised, and the gain** (one run of the code at `8b70ddd`, float64, against the medians above, CPU):

| stage | before [s] | now [s] | what changed |
|---|---|---|---|
| setup | 28.2 | 0.34 | a(s) solved once in float64 on the host for all the parallel skewers (box mode), not once per skewer with diffrax |
| I/O + selection | 299 | 8.7 | threads over the blocks of every file, memory maps, a prefilter of distance bounds on a $256^3$ grid (26.7 million candidates of 1.2 billion), the exact rule on the candidates only: the same cells |
| gas physics | 1.1 / 0.85 | 0.75 / 0.80 | float64 NumPy on the selected cells (float32 lost the Si III densities of metal-poor gas) |
| pieces, default / matched | 43.8 / 10.9 | 20.0 / 9.4 | the Voronoi segments threaded over chunks of skewers, the chords over skewers |
| $\tau$, default | 535 | 32.0 | float32 (2.4$\times$), Harris by default (fourth order for Si III, in place of ExoJAX), a broadcast in `voigt_harris` removed (25$\times$), one line at a time |
| $\tau$, matched | 1362 | 23.5 | fake_spectra's per-cell walk on a window around each cell, batched over the skewers (`core.per_cell`), in place of a host loop over every pixel |
| **total, default / matched** | **908 / 1701** | **61.7 / 42.8** | **15$\times$ / 40$\times$** |

The results did not move beyond float32 rounding and the Harris default: the matched mode stays within rms $\Delta F$
$1.4\times10^{-7}$ ($\mathrm{Ly}\alpha$) and $1.1\times10^{-7}$ (Si III) of fake_spectra, the GPU within $10^{-8}$ (matched) and
$1.2\times10^{-6}$ (default, the float32 rounding of the velocities in the dense sum) of the CPU.

**What remains.** On this target SkewerJAX is 3.3$\times$ (default) and 4.8$\times$ (matched) faster than fake_spectra on the CPU, and
5.9$\times$ and 8.2$\times$ on the GPU, because it reads and processes 0.4 % of the gas. The per-skewer work alone is not yet faster than
fake_spectra's C engine on the CPU: matched pieces + $\tau$ take 33 s against its 19 s (15 s on the GPU). With many more skewers
that work dominates (notebook 14 measures the scaling). Next, by expected gain:
1. **The default pieces** (20 s; 58 % of the default GPU run): `voronoi_segments` samples each dual segment with about
   120 million k-d tree queries; along straight skewers an exact lower envelope of the cells would replace them.
2. **The $\tau$ sum on the CPU** is bound by the Voigt profile: 0.7 ns per evaluation in the dense default sum, 4.8 ns in the
   per-cell sum ($\mathrm{Ly}\alpha$ needs windows of 128–512 pixels to follow the damped wings down to $10^{-7}$). An optional window on
   the dense sum (an accuracy-controlled cut, off by default) would cut the default further; the GPU is 5–7$\times$ faster today.
3. **The selection** (6.6 s): its k-d tree query takes 4.6 s of it; the grid of distance bounds could be refined.
